In [1]:
!pip install pymupdf spacy sentence-transformers faiss-cpu
!python -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable
/home/kaushalparmar/.local/lib/python3.12/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 431.0 kB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import fitz
import re
import json
import spacy
import numpy as np
import faiss

from collections import Counter
from sentence_transformers import SentenceTransformer

/home/kaushalparmar/.local/lib/python3.12/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/kaushalparmar/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-27 21:54:02.749751: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-27 21:54:03.019428: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow w

In [3]:
def extract_clean_text(pdf_path):
    text = []

    with fitz.open(pdf_path) as doc:
        for page in doc:
            blocks = page.get_text("blocks")
            blocks.sort(key=lambda b: (b[1], b[0]))

            for block in blocks:
                block_text = block[4].strip()
                if len(block_text) < 3:
                    continue
                text.append(block_text)

    return "\n".join(text)


def remove_repeated_lines(text):
    lines = text.split("\n")
    freq = Counter(lines)

    cleaned = []
    for line in lines:
        if freq[line] > 5 and len(line.split()) < 3:
            continue
        cleaned.append(line)

    return "\n".join(cleaned)


raw = extract_clean_text("kp_resume.pdf")
final_text = remove_repeated_lines(raw)

print(final_text)

Kaushal Parmar
Phone-Alt +91 99790 09525 | Envelope kaushalparmar018@gmail.com | LINKEDIN LinkedIn
LinkedIn | Github GitHub
GitHub | Map-marker-alt
Rajkot, Gujarat, India
EDUCATION
Marwadi University
2023 – 2027
Bachelor of Technology in Information and Communication Technology
Rajkot, Gujarat, India
SKILLS
Relevant to AR/VR and AI/ML : Unity (Game Development), C#, VR/AR Development, Python, Prompt Engineering (LLM
interaction), OOP, Data Structures and Algorithms, Debugging
Programming Languages : C, C++, Java, R
Frontend Web Technologies : HTML, CSS, Tailwind
Database Technologies : MySQL
Other : Git, Linux, Arduino, NodeMCU, Robotics, 3D Printing, Sensor Interfacing, Embedded Systems
EXPERIENCE AND LEADERSHIP
MADDD Games PVT. LTD.
June 2025 – July 2025
Unity Game Developer Intern
Rajkot, Gujarat, India
– Developed and contributed to live 2D, 3D, and VR projects using Unity Engine, focusing on interactive systems and
user experience
– Gained hands-on experience in professional game 

In [4]:
nlp = spacy.load("en_core_web_sm")

In [5]:
skills_list = [
# Programming Languages
"python","java","c","c++","c#","javascript","typescript","go","rust","kotlin","swift","php","ruby","r","matlab","scala","dart","perl","haskell","objective-c",
# Web Development
"html","css","sass","tailwind","bootstrap","react","angular","vue","next.js","nuxt.js","node.js","express.js","django","flask","spring boot","asp.net","jquery","redux","graphql","rest api",
# Mobile Development
"android development","ios development","react native","flutter","xamarin","jetpack compose","swiftui","kotlin multiplatform",
# Databases
"mysql","postgresql","sqlite","mongodb","firebase","oracle","redis","cassandra","dynamodb","neo4j","elasticsearch","mariadb","cockroachdb",
# Data Science / ML / AI
"machine learning","deep learning","nlp","computer vision","data science","data analysis","data mining","data visualization","pandas","numpy","scikit-learn","tensorflow","keras","pytorch","opencv","xgboost","lightgbm","statsmodels","seaborn","matplotlib",
# Big Data
"hadoop","spark","pyspark","kafka","hive","flink","airflow","databricks",
# DevOps & Cloud
"aws","azure","google cloud","docker","kubernetes","terraform","ansible","jenkins","github actions","gitlab ci","circleci","cloud computing","devops","ci/cd","infrastructure as code",
# Operating Systems & Networking
"linux","unix","windows server","networking","tcp/ip","http","https","dns","dhcp","vpn","firewalls","load balancing","network security",
# Cybersecurity
"cybersecurity","ethical hacking","penetration testing","cryptography","encryption","network security","owasp","siem","malware analysis","reverse engineering",
# Software Engineering Core
"data structures","algorithms","object oriented programming","functional programming","design patterns","system design","low level design","high level design","software development lifecycle","agile","scrum","kanban","debugging","testing","unit testing","integration testing","test automation","clean code","refactoring",
# Tools & Platforms
"git","github","gitlab","bitbucket","jira","confluence","postman","swagger","vscode","intellij","eclipse","android studio","xcode","figma","adobe xd",
# Backend Concepts
"microservices","monolithic architecture","api development","authentication","authorization","jwt","oauth","web sockets","caching","rate limiting",
# Frontend Concepts
"responsive design","ui/ux","web performance","accessibility","seo","cross browser compatibility",
# Embedded / Hardware / IoT
"arduino","raspberry pi","embedded systems","iot","sensor interfacing","microcontrollers","verilog","vhdl","fpga","robotics","3d printing",
# Game Dev / AR-VR
"unity","unreal engine","game development","vr development","ar development","blender","3d modeling","shader programming","physics engine",
# Blockchain / Web3
"blockchain","smart contracts","solidity","web3","ethereum","hyperledger",
# Data Engineering
"etl","data warehousing","data pipelines","snowflake","redshift","bigquery",
# Soft Skills (important for real resumes too)
"problem solving","critical thinking","communication","teamwork","leadership","time management","adaptability","collaboration",
# Misc / Industry Tools
"excel","tableau","power bi","notion","slack","trello","zapier",
# Advanced / Niche
"reinforcement learning","gan","transformers","llm","prompt engineering","computer graphics","parallel computing","distributed systems","edge computing","quantum computing"
]

In [6]:
def extract_skills(text, skills_db):
    text = text.lower()
    found_skills = set()

    for skill in skills_db:
        if re.search(r'\b' + re.escape(skill.lower()) + r'\b', text):
            found_skills.add(skill)

    return list(found_skills)

extracted_skills_list = extract_skills(final_text, skills_list)
print(extracted_skills_list)


['data structures', 'debugging', 'c', 'python', 'css', 'html', 'vr development', 'robotics', 'networking', 'git', 'prompt engineering', 'linux', 'embedded systems', 'tailwind', 'unity', 'algorithms', '3d printing', 'communication', 'leadership', 'mysql', '3d modeling', 'github', 'arduino', 'game development', 'collaboration', 'ui/ux', 'r', 'sensor interfacing', 'llm', 'ar development', 'java']


In [7]:
def extract_projects(text):
    project_section = ""

    pattern = re.compile(r"(projects|project experience)(.*?)(achievements|positions|education|$)", re.S | re.I)
    match = pattern.search(text)

    if match:
        project_section = match.group(2)

    # Split into bullet-like chunks
    projects = re.split(r"\n[-•–]", project_section)
  
    return [p.strip() for p in projects if len(p.strip()) > 30]


extracted_projects = extract_projects(final_text)
print(extracted_projects)

['using Unity Engine, focusing on interactive systems and\nuser experience', 'Gained hands-on experience in professional game development workﬂows, including physics, animation, UI/UX\ndesign, and performance optimization', 'Applied C# programming for scripting interactive elements, enhancing skills in AR/VR application development\nCompetitive Programming Club – Marwadi University\n2025 – Present\nConvener\nRajkot, Gujarat, India', 'Organized Unity Hands-on Workshop: Taught basics of game development, Unity Editor, C# scripting, and guided\nparticipants in recreating interactive games like Flappy Bird', 'Actively contributed to building a stronger tech community by mentoring students in competitive programming, data\nstructures, and algorithms', 'Organized introduction to Git and GitHub workshops for students (conducted twice), improving collaboration skills for\nteam-based projects', 'Uploading daily programming questions to the CP community to foster consistency and problem-solving 

In [8]:
# import re

def is_project_title(line):
    line = line.strip()

    if not line:
        return False

    # Ignore bullets
    if line.startswith(("–", "-", "•")):
        return False

    # Ignore dates
    if re.search(r"\d{4}", line):
        return False

    # Too long = probably description
    if len(line.split()) > 12:
        return False

    # Strong signals
    if "|" in line:
        return True

    # Contains tech keywords
    tech_keywords = [
        "app", "system", "platform", "engine", "simulator",
        "website", "web", "game", "model", "api", "tool"
    ]
    if any(word in line.lower() for word in tech_keywords):
        return True

    # Title case heuristic
    if line.istitle():
        return True

    return False


def extract_projects(text):
    lines = text.split("\n")
    
    projects = []
    in_projects_section = False
    current_project = None

    for line in lines:
        line = line.strip()

        # Start section
        if line.lower() == "projects":
            in_projects_section = True
            continue

        # Stop section
        if in_projects_section and line.lower() in [
            "achievements", "positions of responsibility", "education", "skills"
        ]:
            break

        if not in_projects_section:
            continue

        # Detect title
        if is_project_title(line):
            if current_project:
                projects.append(current_project)

            current_project = {
                "title": line,
                "description": ""
            }
            continue

        # Add description
        if current_project and line.startswith(("–", "-", "•")):
            current_project["description"] += " " + line[1:].strip()

        # Skip noise (dates, empty lines)
        elif re.search(r"\d{4}", line):
            continue

        # Continue description (multi-line wrap)
        elif current_project and line:
            current_project["description"] += " " + line

    # Append last project
    if current_project:
        projects.append(current_project)

    return projects


extracted_projects = extract_projects(final_text)
print(extracted_projects)

[{'title': 'Immersive VR Physiotherapy Rehabilitation Game | Unity VR', 'description': ' Developed an engaging VR application using Unity and C# that gamiﬁes physiotherapy exercises to improve patient adherence and reduce dropout rates through interactive, contextual guidance Integrated real-time motion tracking, progress analytics, and rewarding gameplay mechanics for personalized user feedback in a healthcare context Skills: Unity Engine, C#, VR Development (Oculus Quest), 3D modeling, UX design for interactive systems –'}, {'title': 'Sim Racing Rig | DIY Force-Feedback Racing Simulator', 'description': ' Built a fully custom sim racing cockpit with force-feedback steering, H-pattern gearbox, analog pedals, sequential shifter, and hydraulic handbrake Integrated with games like Assetto Corsa and Forza Horizon 5; showcased at MuFest 2k24, demonstrating real-time'}, {'title': 'interactive systems', 'description': ' Skills: Arduino, C++, sensor interfacing, 3D design & printing, mechanic